In [ ]:
#exhaustive search for CEDRUS-α parameters

import csv
from functools import lru_cache

#tsec,hashbytes = 128,16

#tsec,hashbytes = 192,24

tsec,hashbytes = 256,32



maxsigs=2**64

F = RealField(tsec+100)

if tsec == 128:
    max_size_s = 6960
    max_sign_s = 2128599*2
    max_vrfy_s = 3581
    max_size_f = 17040
    max_sign_f = 91681*2
    max_vrfy_f = 5157
    
if tsec == 192:
    max_size_s = 14784
    max_sign_s = 3265007*2
    max_vrfy_s = 5954
    max_size_f = 35640
    max_sign_f = 145849*2
    max_vrfy_f = 8253

elif tsec == 256:
    max_size_s = 27104
    max_sign_s = 2977382*2
    max_vrfy_s = 10161
    max_size_f = 49696
    max_sign_f = 311762*2
    max_vrfy_f = 8464
    
log2_val = F(2).log()


@lru_cache(maxsize=None)
def qhitprob_cached(qs, r, h):
    leaves = 2**h
    p = F(1 / leaves)
    return binomial(qs, r) * (p**r) * ((1 - p)**(qs - r))

@lru_cache(maxsize=None)
def compute_mincost(h, d):
    y = h % d
    if y == 0:
        h_ary = [h // d] * d
    else:
        h1 = h // d
        h2 = ceil(h / d)
        k = h - h1 * d
        h_ary = [h1] * (d - k) + [h2] * k
    return sum(2 ** i for i in h_ary)

def sec(h, b, k, w2):
    s = F(0)
    base = F(1) - (F(1) / F(2**b)) * F((w2 + 1) / (2 * w2))
    
    for r in range(100):
        p = F(1 - base**r)**k
        s += qhitprob_cached(maxsigs, r, h) * p
    return -F(F(s).log() / log2_val)

def run(l, w):
    s = l * (w - 1) // 2
    ans = 0
    for j in range(s // w + 1):
        term = binomial(l, j) * binomial(s - j * w + l - 1, l - 1)
        if j % 2 == 1:
            ans -= term
        else:
            ans += term
    return ans

def run_ary(l, w_ary, s):
    w_max = max(w_ary)
    w_min = min(w_ary)
    
    if w_max - w_min <= 1:
        c_w = w_ary.count(w_max)
        c_wm1 = l - c_w
        
        ans = 0
        for i in range(c_w + 1):
            for j in range(c_wm1 + 1):
                pwr = i * w_max + j * (w_max - 1)
                if pwr > s:
                    break
                rem = s - pwr
                term = binomial(c_w, i) * binomial(c_wm1, j) * binomial(rem + l - 1, l - 1)
                if (i + j) % 2 == 1:
                    ans -= term
                else:
                    ans += term
        return ans

    dp = [0] * (s + 1)
    dp[0] = 1
    for current_w in w_ary:
        new_dp = [0] * (s + 1)
        current_sum = 0
        for j in range(s + 1):
            current_sum += dp[j]
            if j >= current_w:
                current_sum -= dp[j - current_w]
            new_dp[j] = current_sum
        dp = new_dp
    return dp[s]

def find_w(w, l):
    w_ary = [w] * l
    i = l - 1
    while True:
        s = (sum(w_ary) - l) // 2
        result = run_ary(l, w_ary, s)

        if result < 2**tsec:
            w_ary[(i + 1) % l] += 1
            break
        w_ary[i] -= 1
        i = (i - 1) % l
    return tuple(w_ary)

wots_list = []
w_set = set()
for w in range(8,50):
    for i in range(25,200):
        if run(i,w)>=2**tsec:
            print(w,i)
            w_ary_i = find_w(w,i)
            w_set.add(w_ary_i)
            break
for w_ary in w_set:
    sum_w = sum(w_ary)
    l = len(w_ary)
    diff_sum_l = sum_w - l
    vrfy_cost_wots = (diff_sum_l + 1) // 2
    sign_cost_wots = diff_sum_l // 2 + l
    sign_cost_ht = sum_w + 1
    wots_list.append([w_ary,len(w_ary),sign_cost_wots,sign_cost_ht,vrfy_cost_wots])
                    
with open("CEDRUS-a_256.csv", mode="w") as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['h', 'd', 'b', 'k', 'w1', 'w2', 'l', 'Size', 'sig_speed', 'vrfy_speed', 'sec'])
    for h in range(60, 69):
        for w2 in [1, 2, 4, 8, 16]:
            for b in range(3, 29):
                t = 2**b
                for k in range(5, 70):
                    print(h,w2,b,k)
                    sign_fts = k * (2**b) * (2 + w2) - k + 1
                    vrfy_fts = k * (b + w2)
                    size_part = (1+ h + (b+1)*k)*hashbytes
                    sec_level = sec(h, b, k, w2)
                    if sec_level > tsec:
                        for d in range(5, h):
                            mincost = compute_mincost(h, d)-d 
                            for wots in wots_list:
                                sign_speed = 1 + sign_fts+ mincost * wots[3] + d*wots[2]
                                if sign_speed <= max_sign_f:
                                    sig_size = wots[1] * d * hashbytes + size_part
                                    if sig_size <= max_size_f:
                                        vrfy_speed = vrfy_fts + h + d * wots[4]
                                        if vrfy_speed <= max_vrfy_f:
                                            writer.writerow([h, d, b, k, wots[0], w2, wots[1], sig_size, sign_speed, vrfy_speed, sec_level])
                                    continue
                                if sign_speed <= max_sign_s:
                                    sig_size = wots[1] * d * hashbytes + size_part
                                    if sig_size <= max_size_s:
                                        vrfy_speed = vrfy_fts + h + d * wots[4]
                                        if vrfy_speed <= max_vrfy_s:
                                            writer.writerow([h, d, b, k, wots[0], w2, wots[1], sig_size, sign_speed, vrfy_speed, sec_level])
                        break